<a href="https://colab.research.google.com/github/amaldevk/Sentimental-Analysis/blob/main/Sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [53]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [54]:
df = pd.read_csv("/content/nlp_dataset.csv")
df.head()

,Comment,Emotion
0,i seriously hate one subject to death but now ...,fear
1,im so full of life i feel appalled,anger
2,i sit here to write i start to dig out my feel...,fear
3,ive been really angry with r and i feel like a...,joy
4,i feel suspicious if there is no one outside l...,fear


In [55]:
df.shape

(5937, 2)

In [56]:
df.columns

Index(['Comment', 'Emotion'], dtype='object')

In [57]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5937 entries, 0 to 5936
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Comment  5937 non-null   object
 1   Emotion  5937 non-null   object
dtypes: object(2)
memory usage: 92.9+ KB


In [58]:
df.isnull().sum()

,0
Comment,0
Emotion,0


In [59]:
df['Emotion'].value_counts()

,count
Emotion,
anger,2000
joy,2000
fear,1937


In [60]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [61]:
stop_words = set(stopwords.words('english'))

In [62]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    tokens = word_tokenize(text)
    filtered_tokens = [word for word in tokens if word not in stop_words]
    return " ".join(filtered_tokens)

In [63]:
df['Cleaned_Comment'] = df['Comment'].apply(preprocess_text)
print("\nSample Processed Text:\n", df[['Comment', 'Cleaned_Comment']].head())


Sample Processed Text:
                                              Comment  \
0  i seriously hate one subject to death but now ...   
1                 im so full of life i feel appalled   
2  i sit here to write i start to dig out my feel...   
3  ive been really angry with r and i feel like a...   
4  i feel suspicious if there is no one outside l...   

                                     Cleaned_Comment  
0  seriously hate one subject death feel reluctan...  
1                         im full life feel appalled  
2  sit write start dig feelings think afraid acce...  
3  ive really angry r feel like idiot trusting fi...  
4  feel suspicious one outside like rapture happe...  


Lowercasing (text.lower()): Converts all letters to lowercase so the model treats "Happy", "HAPPY", and "happy" as the exact same word instead of three different ones.

Removing Punctuation (re.sub(...)): Strips away numbers, commas, and special symbols. Things like ! or ? don't carry word meanings, so removing them cleans up extra clutter.

Tokenization (word_tokenize(...)): Splits a long sentence string into a clean list of individual words (tokens) so the computer can process them one by one.

Removing Stopwords (word not in stop_words): Drops non-informative filler words like "is", "the", and "and". This leaves behind only emotional keywords (like "furious", "terrified", or "joyful"), which helps the model focus on what actually indicates feeling.

In [64]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

X_train, X_test, y_train, y_test = train_test_split(
    df['Cleaned_Comment'],
    df['Emotion'],
    test_size=0.2,
    random_state=42,
    stratify=df['Emotion']
)

In [65]:
tfidf = TfidfVectorizer()

In [66]:
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [67]:
X_train_tfidf.shape

(4749, 7733)

Machine learning algorithms cannot read raw strings of text—they only understand numerical values. The TfidfVectorizer converts cleaned text comments into a matrix of numbers by evaluating the importance of each word across the dataset using two main components:



1.   TF (Term Frequency): Measures how often a specific word appears inside a single comment.
2.   IDF (Inverse Document Frequency): Measures how unique or rare a word is across all comments in the dataset.



In [68]:
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
nb_preds = nb_model.predict(X_test_tfidf)

In [69]:
from sklearn.svm import SVC

svm_model = SVC(kernel='linear', random_state=42)
svm_model.fit(X_train_tfidf, y_train)
svm_preds = svm_model.predict(X_test_tfidf)

In [71]:
from sklearn.metrics import (accuracy_score,f1_score,classification_report,confusion_matrix)

print("=== NAIVE BAYES PERFORMANCE ===")
print(f"Accuracy: {accuracy_score(y_test, nb_preds):.4f}")
print(f"Macro F1-Score: {f1_score(y_test, nb_preds, average='macro'):.4f}\n")
print(classification_report(y_test, nb_preds))

=== NAIVE BAYES PERFORMANCE ===
Accuracy: 0.8906
Macro F1-Score: 0.8905

              precision    recall  f1-score   support

       anger       0.88      0.91      0.89       400
        fear       0.87      0.90      0.89       388
         joy       0.92      0.86      0.89       400

    accuracy                           0.89      1188
   macro avg       0.89      0.89      0.89      1188
weighted avg       0.89      0.89      0.89      1188



In [73]:
print("=== SUPPORT VECTOR MACHINE PERFORMANCE ===")
print(f"Accuracy: {accuracy_score(y_test, svm_preds):.4f}")
print(f"Macro F1-Score: {f1_score(y_test, svm_preds, average='macro'):.4f}\n")
print(classification_report(y_test, svm_preds))

=== SUPPORT VECTOR MACHINE PERFORMANCE ===
Accuracy: 0.9369
Macro F1-Score: 0.9367

              precision    recall  f1-score   support

       anger       0.95      0.92      0.93       400
        fear       0.93      0.93      0.93       388
         joy       0.93      0.96      0.95       400

    accuracy                           0.94      1188
   macro avg       0.94      0.94      0.94      1188
weighted avg       0.94      0.94      0.94      1188



In [77]:
nb_acc = accuracy_score(y_test, nb_preds)
nb_f1 = f1_score(y_test, nb_preds, average='macro')

svm_acc = accuracy_score(y_test, svm_preds)
svm_f1 = f1_score(y_test, svm_preds, average='macro')

model_comp = pd.DataFrame({
    "Model": ["Naive Bayes", "Support Vector Machine"],
    "Accuracy": [f"{nb_acc * 100:.2f}%", f"{svm_acc * 100:.2f}%"],
    "Macro F1-Score": [f"{nb_f1 * 100:.2f}%", f"{svm_f1 * 100:.2f}%"]
})

model_comp

,Model,Accuracy,Macro F1-Score
0,Naive Bayes,89.06%,89.05%
1,Support Vector Machine,93.69%,93.67%


SVM scored higher across all tests



*   Accuracy: 93.69% (SVM) vs 89.06% (Naive Bayes)
*   Macro F1-Score: 93.67% (SVM) vs 89.05% (Naive Bayes)

A Support Vector Machine (SVM) works by drawing an imaginary line (or boundary) between different groups of data.It looks at the words in a comment and draws clear boundaries to separate anger, fear, and joy.



*   Converting text into TF-IDF features creates thousands of word columns. SVM easily handles huge lists of words without getting confused or making mistakes.
*   Naive Bayes assumes every word is completely independent of the rest. SVM looks at how words work together, which is essential for understanding human emotions.



